# Sentinel-1 coherence and closure phase (HyP3 InSAR-pair jobs)

Computes the same two products as
[`Earsel26_NISAR-main/notebooks/04_calc_coh_phase_closure_NISAR.ipynb`](Earsel26_NISAR-main/notebooks/04_calc_coh_phase_closure_NISAR.ipynb)
computes for NISAR -- but matching `NISAR_SM.ipynb`'s own inversion input, not that
notebook's simpler sequential-pairs-only convention: `NISAR_SM.ipynb`'s cell 13
builds the **full n x n coherence matrix** (every pair, not just consecutive-in-time
ones) and feeds that whole matrix into `calc_SM_coherences`/`invert_sm`. This
notebook does the Sentinel-1 equivalent, for the bursts downloaded by
[`S1_burst_stations.ipynb`](S1_burst_stations.ipynb).

**Why this doesn't just read the raw burst SLCs directly**: unlike NISAR's GSLC
windows (already geocoded, every acquisition on the same map grid), a raw
Sentinel-1 burst SLC is in native radar (range/azimuth) geometry -- two
acquisitions of the same burst from different passes are offset from each other by
however much the orbit differed that day. Real coherence needs sub-pixel
coregistration between every pair first (TOPS-mode bursts especially so, needing
~0.001-pixel azimuth accuracy to avoid phase jumps at burst boundaries) --
independently geolocating the same (lon, lat) in two uncoregistered bursts and
multiplying would give near-zero coherence everywhere, a processing artifact, not
a real result.

So instead, each required date pair is submitted to **HyP3's on-demand InSAR-ISCE
burst processor** (`hyp3_sdk.submit_insar_isce_burst_job`, the same call the
`hyp3_isce2_burst_stack_for_ts_analysis.ipynb` Coldfoot tutorial uses) -- ASF's
backend runs the real ISCE2 TOPS coregistration + Enhanced Spectral Diversity +
DEM-based flattening, and hands back `_corr.tif` (coherence) and
`_wrapped_phase.tif` (**wrapped** interferometric phase), both geocoded to an 80m
grid (confirmed against the Coldfoot tutorial's own downloaded output). Sampling
those two rasters at the station's (lon, lat) and reconstructing
`z_ij = corr_ij * exp(1j * wrapped_phase_ij)` per pair gives everything needed to
compute coherence and closure phase for every pair/triplet, computed properly.

**Wrapped phase, not unwrapped**: closure phase's entire diagnostic value is
detecting when phase *fails to close* in the wrapped domain -- a real physical
signal (sub-cycle phase bias from a scattering-mechanism change, the De Zan/Ansari
et al. mechanism the inversion is built on). Unwrapped phase would trivially
collapse any well-behaved loop to ~0, since standard unwrapping resolves each
interferogram's own 2*pi ambiguity independently, with no awareness of the
triplet-closure constraint -- it erases the exact signal being measured, it
doesn't reveal it. Only `_wrapped_phase.tif` is used here.

**Every pair, not a reduced sequential-only subset**: for N dates in a
(station, orbit) group, this submits **all N(N-1)/2 pairs**, not just sequential
`(i, i+1)` ones -- matching `NISAR_SM.ipynb`'s full-matrix input exactly (once
built, computing every C(N,3) triplet's closure phase from that full pair set is
free local math -- no extra HyP3 jobs -- so this reports every triplet, not just a
sequential-sliding subset, letting downstream filtering on `min_coherence_VV`
decide what's usable).

:::{important} Real cost and time
For this project's current Alaska station/orbit set, the full pairwise matrix is
**384 required pairs** across 22 (station, orbit) groups (vs. ~210 for a
reduced sequential-only set) -- real, credit-consuming HyP3 jobs, typically tens
of minutes of processing each on ASF's backend, so a full run is realistically many
hours. Section 3 prints the exact job count and asks for confirmation *before*
submitting anything, and Section 4 is designed to be re-run repeatedly (it only
submits/downloads what isn't already done) rather than blocking on one long wait --
the Coldfoot tutorial's own `hyp3.watch()` timed out after ~3 hours on just 10
jobs, so this deliberately polls instead.
:::

:::{note} Unverified assumption
`z_ij = corr_ij * exp(1j * wrapped_phase_ij)` assumes HyP3's wrapped_phase encodes
`angle(reference * conj(secondary))` with `reference` = the chronologically earlier
granule (`granule1` below) -- the standard InSAR convention, matching how
`granule1`/`granule2` are assigned per pair here, but not checked against a real
downloaded product's sign. A flipped convention would negate every closure-phase
value uniformly (coherence magnitude is unaffected either way) -- worth a quick
sanity check against the Coldfoot tutorial's own output once real S1 pairs come
back.
:::

In [ ]:
import sys
import os
import json
from pathlib import Path

import pyproj
pyproj.datadir.set_data_dir(os.path.join(sys.prefix, "Library", "share", "proj"))


def require_globals(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            f"Missing {missing} -- run the earlier setup cells first "
            "(the kernel may have been restarted)."
        )


BASE = Path.cwd()
EARSEL_NOTEBOOKS = BASE / "Earsel26_NISAR-main" / "notebooks"
S1_DIR = BASE / "S1_burst_data_alaska"


## 1. Load the station manifest and determine required pairs

`S1_burst_data_alaska/station_manifest.json` (written by `S1_burst_stations.ipynb`)
maps each station and kept orbit to its matched (NISAR date, S1 date, burst
file_id) triples. Pairs/triplets here are formed by **S1 acquisition date**
(the real chronology), not the NISAR date each entry happened to be matched to.
Every pair within a group is required (see the intro above), matching
`NISAR_SM.ipynb`'s own full-matrix inversion input.

In [2]:
require_globals("BASE", "S1_DIR")

import itertools
import pandas as pd

manifest_path = S1_DIR / "station_manifest.json"
with open(manifest_path) as f:
    manifest = json.load(f)

stations_path = EARSEL_NOTEBOOKS / "ismn_data" / "alaska" / "stations.geojson"
with open(stations_path) as f:
    stations_geojson = json.load(f)
stations = {
    feat["properties"]["station_val"]: (
        feat["geometry"]["coordinates"][1],  # lat
        feat["geometry"]["coordinates"][0],  # lon
    )
    for feat in stations_geojson["features"]
}


def _group_dates(entries):
    """entries: list of {"nisar_date", "s1_date", "file_id"} dicts (one manifest
    group) -> that group's entries sorted by actual S1 acquisition date."""
    return sorted(entries, key=lambda e: e["s1_date"])


# required_pairs: every (i, j), i < j -- the full pairwise coherence matrix,
# matching NISAR_SM.ipynb's own cell 13 (which builds a full n x n coh_full_ds/
# ph_ds matrix from every pair, not just sequential ones, and feeds that whole
# matrix into calc_SM_coherences/invert_sm). Unlike NISAR's GSLC pixels (already
# downloaded locally, so extra pairs cost nothing), every entry here is a
# separately-billed HyP3 job -- see the cost note in the intro above.
required_pairs = []
group_sequences = {}  # (station, orbit) -> sorted list of {date, file_id}
for station, orbits in manifest.items():
    for orbit, entries in orbits.items():
        seq = _group_dates(entries)
        group_sequences[(station, orbit)] = seq
        n = len(seq)
        for i, j in itertools.combinations(range(n), 2):
            required_pairs.append((station, orbit, seq[i]["s1_date"], seq[j]["s1_date"],
                                    seq[i]["file_id"], seq[j]["file_id"]))

print(f"{len(group_sequences)} (station, orbit) group(s)")
print(f"{len(required_pairs)} required pair(s) (full N(N-1)/2 matrix per group)")


22 (station, orbit) group(s)
384 required pair(s) (full N(N-1)/2 matrix per group)


## 2. HyP3 login

In [3]:
import hyp3_sdk as sdk

hyp3 = sdk.HyP3(prompt="password")
PROJECT_NAME = "alaska_s1_coherence"


## 3. Submit missing jobs

Checks `hyp3.find_jobs(name=PROJECT_NAME)` first and only submits pairs that
aren't already submitted (in any status) -- safe to re-run after an
interruption, a kernel restart, or to pick up newly-downloaded S1 dates later,
without resubmitting (and re-billing) work already in flight or done.

In [4]:
require_globals("required_pairs", "hyp3", "PROJECT_NAME")

existing_jobs = hyp3.find_jobs(name=PROJECT_NAME)
existing_pairs = {tuple(j.job_parameters["granules"]) for j in existing_jobs}
print(f"{len(existing_jobs)} job(s) already submitted under '{PROJECT_NAME}'")

to_submit = [p for p in required_pairs if (p[4], p[5]) not in existing_pairs]
print(f"{len(to_submit)}/{len(required_pairs)} pair(s) still need submitting")

if to_submit:
    print(f"\nAbout to submit {len(to_submit)} real, credit-consuming HyP3 jobs.")
    _confirm = input("Type 'yes' to proceed: ")
    if _confirm.strip().lower() == "yes":
        jobs = sdk.Batch()
        for station, orbit, date_i, date_j, file_id_i, file_id_j in to_submit:
            jobs += hyp3.submit_insar_isce_burst_job(
                granule1=file_id_i,
                granule2=file_id_j,
                name=PROJECT_NAME,
                apply_water_mask=True,
                looks="20x4",  # 80m, matching the window footprint used throughout this project
            )
        print(f"Submitted {len(jobs)} job(s)")
    else:
        print("Skipped submission.")


384 job(s) already submitted under 'alaska_s1_coherence'
0/384 pair(s) still need submitting


## 4. Check status, download, and extract completed jobs

Re-run this cell any time to pick up newly-finished jobs -- already-extracted
pairs are skipped (checked by directory existence), so this is cheap to poll
repeatedly instead of blocking on one long wait.

In [5]:
require_globals("hyp3", "PROJECT_NAME", "S1_DIR")

all_jobs = hyp3.find_jobs(name=PROJECT_NAME)
n_succeeded = sum(1 for j in all_jobs if j.succeeded())
n_running = sum(1 for j in all_jobs if j.running())
n_pending = sum(1 for j in all_jobs if j.pending())
n_failed = sum(1 for j in all_jobs if j.failed())
print(f"{len(all_jobs)} total: {n_succeeded} succeeded, {n_running} running, "
      f"{n_pending} pending, {n_failed} failed")

coh_dir = S1_DIR / "coh_phase_closure"
coh_dir.mkdir(exist_ok=True)

# pair_products: (file_id_i, file_id_j) -> path to the extracted product directory
pair_products = {}
for path in coh_dir.iterdir():
    if path.is_dir() and (path / f"{path.name}.txt").exists():
        with open(path / f"{path.name}.txt") as f:
            meta = dict(line.strip().split(": ", 1) for line in f if ": " in line)
        ref = meta.get("Reference Granule")
        sec = meta.get("Secondary Granule")
        if ref and sec:
            pair_products[(ref, sec)] = path

n_new = 0
for job in all_jobs:
    if not job.succeeded():
        continue
    granules = tuple(job.job_parameters["granules"])
    if granules in pair_products:
        continue
    zip_paths = job.download_files(str(coh_dir))
    for zip_path in zip_paths:
        product_dir = sdk.util.extract_zipped_product(zip_path)
    pair_products[granules] = Path(product_dir)
    n_new += 1

print(f"{n_new} newly extracted, {len(pair_products)} pair product(s) available on disk")
if n_failed:
    print(f"\n{n_failed} job(s) failed -- inspect job.logs / the HyP3 dashboard for these granule pairs:")
    for j in all_jobs:
        if j.failed():
            print(f"  {j.job_parameters['granules']}")


384 total: 384 succeeded, 0 running, 0 pending, 0 failed
0 newly extracted, 384 pair product(s) available on disk


## 5. Sample coherence/phase at each station, build the coherence and closure-phase CSVs

For each pair product available on disk: sample `_corr.tif` (coherence) and
`_wrapped_phase.tif` at the station's own pixel (nearest, single pixel -- the
product is already at 80m/pixel, matching this project's window-footprint
convention, so no further windowing is needed). Reconstructs
`z_ij = corr_ij * exp(1j * wrapped_phase_ij)` per pair, then reports **every**
pair's coherence and **every** triplet's closure phase (computing a triplet's
closure is free local math once its three legs' pairs are on disk -- no extra
HyP3 cost), with column names matching `nisar_coherence_closure.py`'s NISAR
output (`_VV` in place of `_HH`/`_HV`, since these bursts are VV-only) so
downstream code can treat NISAR and S1 outputs symmetrically.

**Also saves `phase_VV` and `local_incidence_angle_deg` on the coherence CSV**
(both previously dropped here) -- an SM inversion notebook needs
`coherence_VV * exp(i * phase_VV)` to reconstruct `ph_DS` (same convention
`nisar_coherence_closure.py` uses for NISAR) and an incidence angle for
`invert_sm`'s `inc_DS`, neither of which this section used to persist (it only
kept the closure-phase math's *intermediate* `z_ij`, not `angle(z_ij)` itself,
and never touched incidence at all). `phase_VV` is `angle(z_ij)` per pair.
Incidence angle is derived from each product's own `_lv_theta.tif` (look-vector
elevation: the angle between the horizontal surface and the sensor look
vector) as `90 - degrees(lv_theta)` -- the angle from vertical/nadir that
`local_incidence_angle_deg` means elsewhere in this project.

In [6]:
require_globals("pair_products", "group_sequences", "stations")

import itertools

import numpy as np
import rasterio
import pyproj as _pyproj

output_dir = BASE / "S1_coherence_closure_alaska"
output_dir.mkdir(exist_ok=True)


def _sample_pixel(tif_path, lon, lat):
    with rasterio.open(tif_path) as src:
        transformer = _pyproj.Transformer.from_crs(4326, src.crs, always_xy=True)
        x, y = transformer.transform(lon, lat)
        row, col = src.index(x, y)
        if not (0 <= row < src.height and 0 <= col < src.width):
            return np.nan
        val = src.read(1)[row, col]
        return float(val) if val != src.nodata else np.nan


def _pair_z(product_dir, lon, lat):
    """(coherence, z, incidence_deg) at the station pixel for one HyP3
    InSAR-pair product, or (nan, nan+0j, nan) if the coherence/phase rasters
    have no valid data there. `z = coherence * exp(1j * wrapped_phase)`.
    Incidence angle comes from `_lv_theta.tif` (look-vector elevation, angle
    from horizontal) as `90 - degrees(lv_theta)` (angle from vertical/nadir)
    -- missing separately from the coherence/phase rasters sometimes, so its
    own NaN doesn't invalidate coh/z."""
    corr = _sample_pixel(product_dir / f"{product_dir.name}_corr.tif", lon, lat)
    phase = _sample_pixel(product_dir / f"{product_dir.name}_wrapped_phase.tif", lon, lat)
    lv_theta = _sample_pixel(product_dir / f"{product_dir.name}_lv_theta.tif", lon, lat)
    inc_deg = 90.0 - np.degrees(lv_theta) if not np.isnan(lv_theta) else np.nan
    if np.isnan(corr) or np.isnan(phase):
        return np.nan, np.nan + 0j, inc_deg
    return corr, corr * np.exp(1j * phase), inc_deg


for (station, orbit), seq in group_sequences.items():
    lat, lon = stations[station]
    n = len(seq)

    pair_cache = {}  # (file_id_i, file_id_j) -> (coherence, z, incidence_deg), memoized per station-orbit run

    def _get_pair(fid_i, fid_j):
        if (fid_i, fid_j) not in pair_cache:
            product_dir = pair_products.get((fid_i, fid_j))
            pair_cache[(fid_i, fid_j)] = (
                _pair_z(product_dir, lon, lat) if product_dir is not None else (np.nan, np.nan + 0j, np.nan)
            )
        return pair_cache[(fid_i, fid_j)]

    coherence_rows = []
    for i, j in itertools.combinations(range(n), 2):
        a, b = seq[i], seq[j]
        coh, z, inc_deg = _get_pair(a["file_id"], b["file_id"])
        coherence_rows.append({
            "date_1": a["s1_date"], "date_2": b["s1_date"],
            "temporal_baseline_days": (
                pd.Timestamp(b["s1_date"]) - pd.Timestamp(a["s1_date"])
            ).total_seconds() / 86400.0,
            "orbit": orbit,
            "coherence_VV": coh,
            "phase_VV": float(np.angle(z)),
            "local_incidence_angle_deg": inc_deg,
        })

    closure_rows = []
    for i, j, k in itertools.combinations(range(n), 3):
        a, b, c = seq[i], seq[j], seq[k]
        coh_ij, z_ij, _ = _get_pair(a["file_id"], b["file_id"])
        coh_jk, z_jk, _ = _get_pair(b["file_id"], c["file_id"])
        coh_ik, z_ik, _ = _get_pair(a["file_id"], c["file_id"])
        triple = z_ij * z_jk * np.conj(z_ik)
        closure_phase = float(np.angle(triple)) if triple != 0 and not np.isnan(triple) else np.nan
        min_coh = np.nanmin([coh_ij, coh_jk, coh_ik])
        closure_rows.append({
            "date_1": a["s1_date"], "date_2": b["s1_date"], "date_3": c["s1_date"],
            "temporal_baseline_12_days": (pd.Timestamp(b["s1_date"]) - pd.Timestamp(a["s1_date"])).total_seconds() / 86400.0,
            "temporal_baseline_23_days": (pd.Timestamp(c["s1_date"]) - pd.Timestamp(b["s1_date"])).total_seconds() / 86400.0,
            "temporal_baseline_13_days": (pd.Timestamp(c["s1_date"]) - pd.Timestamp(a["s1_date"])).total_seconds() / 86400.0,
            "orbit": orbit,
            "closure_phase_VV": closure_phase,
            "min_coherence_VV": min_coh,
        })

    coherence_path = output_dir / f"{station}_orbit{orbit}_S1_coherence.csv"
    closure_path = output_dir / f"{station}_orbit{orbit}_S1_closure_phase.csv"
    pd.DataFrame(coherence_rows).to_csv(coherence_path, index=False)
    pd.DataFrame(closure_rows).to_csv(closure_path, index=False)

    n_coh_ready = sum(1 for r in coherence_rows if not np.isnan(r["coherence_VV"]))
    n_clo_ready = sum(1 for r in closure_rows if not np.isnan(r["closure_phase_VV"]))
    print(f"{station} orbit {orbit}: {n_coh_ready}/{len(coherence_rows)} pair(s) ready, "
          f"{n_clo_ready}/{len(closure_rows)} triplet(s) ready")


AtigunPass orbit 94: 28/28 pair(s) ready, 56/56 triplet(s) ready
Coldfoot orbit 94: 21/21 pair(s) ready, 35/35 triplet(s) ready
EagleSummit orbit 94: 28/28 pair(s) ready, 56/56 triplet(s) ready
EagleSummit orbit 123: 21/21 pair(s) ready, 35/35 triplet(s) ready
EagleSummit orbit 160: 15/15 pair(s) ready, 20/20 triplet(s) ready
GobblersKnob orbit 94: 28/28 pair(s) ready, 56/56 triplet(s) ready
GraniteCrk orbit 94: 28/28 pair(s) ready, 56/56 triplet(s) ready
GraniteCrk orbit 160: 6/6 pair(s) ready, 4/4 triplet(s) ready
ImnaviatCreek orbit 94: 28/28 pair(s) ready, 56/56 triplet(s) ready
LittleChenaRidge orbit 94: 28/28 pair(s) ready, 56/56 triplet(s) ready
LittleChenaRidge orbit 160: 10/10 pair(s) ready, 10/10 triplet(s) ready
McGrath orbit 36: 6/6 pair(s) ready, 4/4 triplet(s) ready
NukaGlacier orbit 131: 10/10 pair(s) ready, 10/10 triplet(s) ready
PilgrimHotSprings orbit 153: 15/15 pair(s) ready, 20/20 triplet(s) ready
RockyPoint orbit 153: 15/15 pair(s) ready, 20/20 triplet(s) ready
Spr

<hr>

## Summary

You now have, under `S1_coherence_closure_alaska/`, two CSVs per (station, orbit)
group: `{station}_orbit{orbit}_S1_coherence.csv` (every pair, `coherence_VV` +
`phase_VV` + `local_incidence_angle_deg`) and
`{station}_orbit{orbit}_S1_closure_phase.csv` (every triplet, `closure_phase_VV` +
`min_coherence_VV`) -- the same schema as `nisar_coherence_closure.py`'s NISAR
output, `_VV` in place of `_HH`/`_HV`, and matching `NISAR_SM.ipynb`'s full-matrix
scope rather than a reduced sequential-only subset. `coherence_VV * exp(i *
phase_VV)` reconstructs the complex interferometric value an SM inversion needs
(`S1_SM_alaska.ipynb`), without touching the raw HyP3 products a second time.

A row with `NaN` means the underlying HyP3 job either hasn't finished yet or
failed -- re-run Sections 3-5 after more jobs complete (Section 3 won't
resubmit anything already in flight or done; Section 5 rebuilds both CSVs from
whatever pair products are on disk, cheap since it's purely local).

<hr>